# LangChain: Memory

## 1) Short-term (state) : InMemorySaver()

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model='gpt-5.4-mini',
    tools=[],
    checkpointer=InMemorySaver()
)

result1 = agent.invoke(
    {
        'messages': [{ 'role': 'user', 'content': '안녕 내이름은 김한호야'}]
    },
    {'configurable': {'thread_id':'hanho'}}
)

In [5]:
print(f'{result1['messages'][-1].content}')

안녕하세요, 김한호님! 반가워요.  
무엇을 도와드릴까요?


In [ ]:
result2 = agent.invoke(
    {
        'messages': [{ 'role': 'user', 'content': '내 이름이 뭐라고했지?'}]
    },
    {'configurable': {'thread_id':'hanho'}}
)

In [8]:
print(f'{result2['messages'][-1].content}')

김한호님이라고 하셨어요.


## 2) Long-term (store) : InMemoryStore()

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

namespace = ('user', 'minsu', 'preference')     # user/minsu/preference 와 같은 디렉토리형태가 된다.

store.put(namespace, 'language', {'value': 'ko'}) # user/minsu/preference/{language:{value:ko}}

result = store.get(namespace, 'language')
result

Item(namespace=['user', 'minsu', 'preference'], key='language', value={'value': 'ko'}, created_at='2026-04-20T01:13:17.202363+00:00', updated_at='2026-04-20T01:13:17.202363+00:00')

In [ ]:
'''
Item(
    namespace=['user', 'minsu', 'preference'],
    key='language',
    value={'value': 'ko'},
    ...
)
'''
result.value['value']

'ko'

### 3) 실전예제

In [12]:
from dataclasses import dataclass
from typing import Optional

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore


# -----------------------------------
# 1. 사용자 구분용 context 정의
# -----------------------------------
@dataclass
class UserContext:
    user_id: str


# -----------------------------------
# 2. 메모리 저장소 생성
# -----------------------------------
store = InMemoryStore()


# -----------------------------------
# 3. 사용자 선호 저장 tool
# -----------------------------------
@tool
def save_preference(category: str, value: str, runtime: ToolRuntime[UserContext]) -> str:
    """
    사용자의 선호 정보를 저장합니다.
    예: category='drink', value='tea'
    """
    user_id = runtime.context.user_id
    namespace = ('users', user_id, 'preferences')
    
    runtime.store.put(
        namespace,
        category,
        {"value":value}
    )
    
    return f"{category} 선호 정보에 '{value}'를 저장했습니다."


# -----------------------------------
# 4. 사용자 선호 조회 tool
# -----------------------------------
@tool
def load_preference(category: str, runtime: ToolRuntime[UserContext]) -> str:
    """
    사용자의 선호 정보를 불러옵니다.
    """
    user_id = runtime.context.user_id
    namespace = ('users', user_id, 'preferences')
    
    memory = runtime.store.get(namespace, category)
    
    if memory is None:
        return f"{category}에 대한 저장된 정보가 없습니다."
    
    return f"{category}에 대한 저장 정보는 '{memory.value['value']}'입니다."


# -----------------------------------
# 5. 선호 기반 추천 tool
# -----------------------------------
@tool
def recommend_by_preference(runtime: ToolRuntime[UserContext]) -> str:
    """저장된 선호를 바탕으로 간단한 추천을 합니다."""
    
    user_id = runtime.context.user_id
    namespace = ('users', user_id, 'preferences')
    
    drink_memory = runtime.store.get(namespace, 'drink')
    food_memory = runtime.store.get(namespace, 'food')
    
    drink = drink_memory.value['value'] if drink_memory else None
    food = food_memory.value['value'] if food_memory else None
    
    if drink and food:
        return f"당신은 {drink}와 {food}를 좋아하므로, {food}와 함께 {drink}를 추천합니다."
    elif drink:
        return f"당신은 {drink}를 좋아하므로, 오늘은 {drink}를 추천합니다."
    elif food:
        return f"당신은 {food}를 좋아하므로, {food} 관련 메뉴를 추천합니다."
    else:
        return "저장된 선호 정보가 없어서 아직 추천하기 어렵습니다."


In [15]:
agent = create_agent(
    'openai:gpt-5.4-mini',
    tools=[save_preference, load_preference, recommend_by_preference],
    store=store
)

In [16]:
user_context = UserContext(user_id='hanho')

response1 = agent.invoke(
    {
        'messages': [{'role': 'user', 'content': '나는 김한호로, 음료는 tea를 좋아하고 음식은 파스타를 좋아해.'}]
    },
    context=user_context
)

In [19]:
print(f"{response1['messages'][-1].content}")

기억해둘게요, 김한호님.

- 음료 선호: tea
- 음식 선호: pasta


In [20]:
response2 = agent.invoke(
    {
        'messages': [{'role': 'user', 'content': '내 이름은 뭐고 어떤 음식을 좋아한다고 했지?'}]
    },
    context=user_context
)

In [21]:
print(f"{response2['messages'][-1].content}")

저장된 정보 기준으로는 **이름은 아직 저장되어 있지 않고**, 좋아하는 음식은 **pasta**예요.
